# 03 · Ranking de estados y participación en el consumo total · Resolución

Solución comentada de esta práctica de livecoding.

Seleccioná el kernel de `.venv` y usá **Reiniciar kernel y ejecutar todo**. La sección 3 contiene la función resuelta y la sección 4 ejecuta las pruebas de la práctica junto con la comprobación sobre datos reales. El resultado completo queda en la variable `resultado`.

La carga reutiliza `practicas_livecoding/preparacion.py` y los datasets ya descargados en `practicas_livecoding/datos/`. Conservá ambas carpetas dentro del mismo proyecto.

## 1. Preparación lista

La base equivale a `base` del notebook original: fecha como índice, nombres preparados y faltantes, negativos y ceros conservados.

Los archivos de `practicas_livecoding/datos/` ya están descargados. Si faltan, se descargan una vez desde Kaggle y se guardan localmente.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from pandas.testing import assert_frame_equal

# Encuentra la carpeta tanto desde la raíz del proyecto como desde este notebook.
ubicaciones = [Path.cwd(), *Path.cwd().parents]
carpeta = next(
    (
        candidata
        for ubicacion in ubicaciones
        for candidata in [ubicacion, ubicacion / "practicas_livecoding"]
        if candidata.name == "practicas_livecoding"
        and (candidata / "preparacion.py").is_file()
    ),
    None,
)
if carpeta is None:
    raise FileNotFoundError("Abrí el notebook dentro de la carpeta practicas_livecoding.")

if str(carpeta) not in sys.path:
    sys.path.insert(0, str(carpeta))

from preparacion import cargar_viviendas, cargar_energia

pd.set_option("display.max_columns", 20)

In [2]:
datos = cargar_energia()

print(f"Filas: {len(datos):,} | Columnas: {len(datos.columns)}")
display(datos.head())

Filas: 285,499 | Columnas: 13


,TipoConsumidor,Sistema,UF,Classe,Setor Econômico - N2,Setor Econômico - N3,Tipo Tensão - N1,Tipo Tensão - N2,Tipo Tensão - N3,Faixa de Consumo N1,Faixa de Consumo N2,Consumidores,Consumo
Data,,,,,,,,,,,,,
2014-01-01,Cativo,Sudeste / Centro-Oeste,RO,Residencial,Convencional (Excepto Baixa Renda),TOTAL,A - Alta Tensão,TOTAL,TOTAL,Alta Tensão,Alta Tensão,1.0,8.0
2014-01-01,Cativo,Sudeste / Centro-Oeste,SP,Rural,Serviço Público de Irrigação,TOTAL,A - Alta Tensão,"A-4 - 2,3 a 25 kV",TOTAL,Não Aplicável,Não Aplicável,6.0,145.0
2014-01-01,Cativo,Sudeste / Centro-Oeste,SP,Rural,Serviço Público de Irrigação,TOTAL,B - Baixa Tensão,TOTAL,TOTAL,Não Aplicável,Não Aplicável,31.0,18.0
2014-01-01,Cativo,Sudeste / Centro-Oeste,MS,Residencial,Baixa Renda,Baixa Renda N2,B - Baixa Tensão,TOTAL,TOTAL,Baixa Renda,> 200 kWh,44193.0,13171.0
2014-01-01,Cativo,Sistemas Isolados,AC,Residencial,Convencional (Excepto Baixa Renda),TOTAL,B - Baixa Tensão,TOTAL,TOTAL,Convencional,0-30 kWh,5216.0,149.0


## 2. Consigna y resultado esperado

Sumá el `Consumo` de todo el período por `UF` y devolvé los `n` estados con mayor total.

- Usá `sum(min_count=1)` y omití del ranking las UF sin ninguna medición de consumo.
- Devolvé un DataFrame con índice `UF` y columnas `consumo_total`, `participacion_pct`.
- La participación es el consumo de la UF dividido por el total de **todas las UF con datos**,
  multiplicado por 100. No uses solo la suma del top `n` como denominador.
- Ordená por consumo descendente; si hay empate, por nombre de UF ascendente.
- Conservá los valores negativos. Si el total general es cero, las participaciones son `NaN`.
- `n` es un entero positivo. Si hay menos UF, devolvé las disponibles.
  No redondees ni modifiques la entrada.

## 3. Resolución comentada

1. Sumamos por UF y descartamos los grupos sin ninguna medición de consumo.
2. Calculamos las participaciones con el total general antes de aplicar `head(n)`.
3. Ordenamos alfabéticamente y luego por consumo con orden estable para resolver los empates.

Si el total general es cero, la participación no está definida y queda como `NaN`.
Las participaciones del top cinco no tienen por qué sumar 100 %.

In [3]:
def ranking_estados(datos, n=5):
    """Ordena las UF por consumo y calcula su participación en el total general."""
    consumo_por_uf = datos.groupby("UF")["Consumo"].sum(min_count=1).dropna()
    resultado = consumo_por_uf.to_frame("consumo_total")

    # El denominador incluye todas las UF con datos, antes de seleccionar el top.
    total_general = consumo_por_uf.sum()
    if total_general == 0:
        resultado["participacion_pct"] = np.nan
    else:
        resultado["participacion_pct"] = (
            resultado["consumo_total"] / total_general * 100
        )

    # Orden alfabético previo y orden estable por consumo para resolver empates.
    resultado = resultado.sort_index()
    resultado = resultado.sort_values(
        "consumo_total", ascending=False, kind="stable"
    )
    return resultado.head(n)

## 4. Pruebas y ejecución real

Se conservan los casos y las comprobaciones de la práctica original. Los mensajes `OK` confirman que pasan y que la entrada permanece intacta. La tabla muestra las primeras 20 filas; `resultado` contiene la salida completa.

In [4]:
# Estas pruebas no reemplazan tu función: comprueban su resultado.
def comparar(actual, esperado):
    assert isinstance(actual, pd.DataFrame), "La función debe devolver un DataFrame."
    assert_frame_equal(
        actual, esperado,
        check_dtype=False, check_index_type=False, check_column_type=False,
        check_freq=False, rtol=1e-6, atol=1e-8,
    )


def probar(funcion):
    caso = pd.DataFrame({
        "UF": ["SP", "RJ", "SP", "MG", "RS", "SC", "RS"],
        "Consumo": [100, 150, 50, 50, np.nan, -10, np.nan],
    })
    original = caso.copy(deep=True)
    esperado = pd.DataFrame({
        "consumo_total": [150, 150, 50, -10],
        "participacion_pct": [150 / 340 * 100, 150 / 340 * 100, 50 / 340 * 100, -10 / 340 * 100],
    }, index=pd.Index(["RJ", "SP", "MG", "SC"], name="UF"))
    comparar(funcion(caso, n=2), esperado.iloc[:2])
    comparar(funcion(caso, n=99), esperado)
    comparar(funcion(caso), esperado)

    total_cero = pd.DataFrame({"UF": ["SP", "RJ"], "Consumo": [10, -10]})
    esperado_cero = pd.DataFrame({
        "consumo_total": [10, -10], "participacion_pct": [np.nan, np.nan],
    }, index=pd.Index(["SP", "RJ"], name="UF"))
    comparar(funcion(total_cero), esperado_cero)

    sin_datos = pd.DataFrame({"UF": ["SP"], "Consumo": [np.nan]})
    comparar(funcion(sin_datos), esperado.iloc[:0])
    comparar(caso, original)
    print("OK: sumas, empates, denominador global, negativos, n variable y total cero.")


probar(ranking_estados)

# Comprobamos también que el dataset real permanezca intacto.
antes = datos.copy(deep=True)
resultado = ranking_estados(datos)
comparar(datos, antes)
assert len(resultado) == min(5, datos.loc[datos["Consumo"].notna(), "UF"].nunique())
assert resultado["consumo_total"].is_monotonic_decreasing
np.testing.assert_allclose(
    resultado["participacion_pct"], resultado["consumo_total"] / datos["Consumo"].sum() * 100,
)
print(f"OK: dataset real. Resultado: {len(resultado):,} filas.")
display(resultado.head(20))

OK: sumas, empates, denominador global, negativos, n variable y total cero.
OK: dataset real. Resultado: 5 filas.


,consumo_total,participacion_pct
UF,,
SP,1.335301e+09,27.533642
MG,5.702171e+08,11.757763
RJ,3.912556e+08,8.067612
PR,3.248535e+08,6.698415
RS,3.022107e+08,6.231525


## Para repasar

Volvé a la práctica sin resolver e intentá escribir la función desde cero. Explicá qué representa cada fila de la salida, cómo se resuelven los casos límite y por qué el cálculo respeta la consigna.